In [ ]:
import os
import re
import json
import time
import tiktoken

from IPython.core.display_functions import clear_output
from dotenv import load_dotenv
from pathlib import Path
from tuutrag.data import D
from tuutrag.types import *
from tuutrag.prompt import create_batch_request
from tuutrag.prompts.manager import load_prompt
from tuutrag.utils import count_batch_request_tokens
from openai import OpenAI
from tqdm import tqdm

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
load_dotenv()

OPENAI_MODEL: str = "gpt-4.1-mini"

openai_key: str | None = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("OPENAI_API_KEY is not set in environment variables.")

openai_client: OpenAI = OpenAI(api_key=openai_key)

In [ ]:
with open(D.find("fixed_size_chunks.json"), "r", encoding="utf-8") as f:
    all_chunks: list[BranchChunk] = json.load(f)

seen: set[str] = set()
unique_branch_chunks: list[BranchChunk] = []
for obj in all_chunks:
    if obj["uuid"] not in seen:
        seen.add(obj["uuid"])
        unique_branch_chunks.append(obj)

In [ ]:
SCHEME: str = "o200k_base"
enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

total_tokens: int = sum(len(enc.encode(obj["chunk"])) for obj in unique_branch_chunks)
print(f"{total_tokens:,}")

In [ ]:
all_requests: list[BatchRequest] = [
    create_batch_request(
        custom_id=chunk["uuid"],
        model=OPENAI_MODEL,
        **load_prompt("leaf.j2", text=chunk["chunk"])
    )
    for chunk in tqdm(unique_branch_chunks, desc="Creating Batch Requests")
]

token_counts: list[int] = [
    count_batch_request_tokens(
        enc=enc,
        payload=req
    ) for req in tqdm(all_requests, desc="Counting Total Tokens")
]

In [ ]:
# Load already-processed UUIDs from leafs.json
with open(D.find("leafs.json"), "r", encoding="utf-8") as f:
    processed_ids: set[str] = {
        obj["uuid"].rsplit(".", 1)[0] for obj in json.load(f)
    }

# Filter to only unprocessed requests
pending_requests: list[BatchRequest] = [
    req for req in all_requests if req["custom_id"] not in processed_ids
]
pending_token_counts: list[int] = [
    token_counts[i] for i, req in enumerate(all_requests)
    if req["custom_id"] not in processed_ids
]

print(f"Total requests    : {len(all_requests):,}")
print(f"Already processed : {len(processed_ids):,}")
print(f"Pending           : {len(pending_requests):,}")

In [ ]:
all_requests = pending_requests
token_counts = pending_token_counts

In [ ]:
MAX_TOKENS_PER_BATCH: int = 1_000_000  # OpenAI batch limit (w/ safety padding)

batches: list[list[BatchRequest]] = []
current_batch: list[BatchRequest] = []
current_tokens: int = 0

for req, req_tokens in zip(all_requests, token_counts):
    if current_tokens + req_tokens > MAX_TOKENS_PER_BATCH and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_tokens = 0
    current_batch.append(req)
    current_tokens += req_tokens

if current_batch:
    batches.append(current_batch)

print(f"Total requests : {len(all_requests):,}")
print(f"Total batches  : {len(batches):,}")
for i, batch in enumerate(batches):
    batch_tokens = sum(token_counts[all_requests.index(req)] for req in batch)
    print(f"  Batch {i + 1}: {len(batch):,} requests | {batch_tokens:,} tokens")

In [ ]:
POLL_INTERVAL: int = 30
RETRY_WAIT:    int = 120
MAX_RETRIES:   int = 5

metadata_path: Path = D.leaf_batches / "batch_metadata.json"
master_file:   Path = D.processed    / "leafs.json"

In [ ]:
def _save_metadata(metadata: list[dict]) -> None:
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

def _get_metadata() -> list[dict]:
    with open(metadata_path, "r", encoding="utf-8") as f:
        return json.load(f)

def _remove_from_metadata(batch_id: str) -> None:
    metadata = _get_metadata()
    _save_metadata([m for m in metadata if m.get("id") != batch_id])

In [ ]:
def _append_leaves(output_text: str) -> int:
    """Parse <|> delimited LLM output and append leaves to master file. Returns count added."""
    responses: list[dict] = [
        json.loads(line)
        for line in output_text.strip().split("\n")
        if line.strip()
    ]

    with open(master_file, "r", encoding="utf-8") as f:
        existing: list[dict] = json.load(f)

    added: int = 0
    for response in responses:
        res_custom_id: str = response["custom_id"]
        res_content: str   = response["response"]["body"]["choices"][0]["message"]["content"]
        leaves: list[str]  = [leaf.strip() for leaf in res_content.split("<|>") if leaf.strip()]
        for i, leaf in enumerate(leaves, start=1):
            existing.append({"uuid": f"{res_custom_id}.{i}", "text": leaf})
            added += 1

    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    return added

In [ ]:
def _resubmit(old_batch_id: str) -> str:
    """Resubmit a failed batch from its original .jsonl and update metadata. Returns new batch_id."""
    metadata  = _get_metadata()
    entry     = next(m for m in metadata if m["id"] == old_batch_id)

    with open(D.leaf_batches / entry["file_name"], "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    new_job = openai_client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    entry["id"] = new_job.id
    _save_metadata(metadata)

    return new_job.id

In [ ]:
pattern = re.compile(r"^leaf_batch_\d+\.jsonl$")

# Delete matched .jsonl files locally
deleted_local = [
    f for f in metadata_path.parent.iterdir()
    if f.is_file() and pattern.match(f.name) and not f.unlink()
]
print(f"Deleted locally  : {len(deleted_local):,} .jsonl files")

# Delete input files and cancel batch jobs on OpenAI
deleted_files  = 0
cancelled_jobs = 0

try:
    metadata = _get_metadata()
    for entry in metadata:
        # Delete uploaded input file
        file_id = entry.get("input_file_id")
        if file_id:
            try:
                openai_client.files.delete(file_id)
                deleted_files += 1
                print(f"  Deleted file      : {file_id}  ({entry['file_name']})")
            except Exception as e:
                print(f"  Could not delete file {file_id}: {e}")

        # Cancel batch job
        batch_id = entry.get("id")
        if batch_id:
            try:
                openai_client.batches.cancel(batch_id)
                cancelled_jobs += 1
                print(f"  Cancelled batch   : {batch_id}")
            except Exception as e:
                print(f"  Could not cancel batch {batch_id}: {e}")

except FileNotFoundError:
    print("  No batch_metadata.json found — skipping remote cleanup.")

print(f"Deleted files    : {deleted_files:,}")
print(f"Cancelled jobs   : {cancelled_jobs:,}")

# Delete batch_metadata.json
if metadata_path.exists():
    metadata_path.unlink()
    print("Deleted          : batch_metadata.json")

In [ ]:
batch_file_ids: list[dict] = []
batch_metadata: list[dict] = []

for idx, batch in enumerate(tqdm(batches, desc="Uploading batch files")):
    file_name: str = f"leaf_batch_{idx}.jsonl"

    with open(metadata_path.parent / file_name, "w", encoding="utf-8") as f:
        for request in batch:
            f.write(json.dumps(request) + "\n")

    with open(metadata_path.parent / file_name, "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    batch_file_ids.append({
        "file_name": file_name,
        "input_file_id": batch_file.id,
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(batch_file_ids, f, indent=2)

print(f"\nUploaded {len(batch_file_ids):,} files. Ready to submit jobs in Monitor cell.")

In [ ]:
print(f"Submitting and processing {len(batch_file_ids)} batch(es) one at a time.\n")

for file_entry in batch_file_ids:
    file_name: str = file_entry["file_name"]
    input_file_id: str = file_entry["input_file_id"]
    retries: int = 0

    batch_job = openai_client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    batch_id: str = batch_job.id

    metadata = _get_metadata()
    for m in metadata:
        if m["input_file_id"] == input_file_id:
            m.update({"id": batch_id, **batch_job.model_dump()})
    _save_metadata(metadata)

    print(f"{file_name} | submitted: {batch_id}")

    while True:
        job = openai_client.batches.retrieve(batch_id)
        status = job.status

        if status == "completed":
            output = openai_client.files.content(job.output_file_id)
            added = _append_leaves(output.text)
            _remove_from_metadata(batch_id)
            print(f"Complete - {added} leaves added | {len(_get_metadata())} remaining\n")
            break

        elif status == "failed":
            errors = job.errors.data if job.errors else []
            token_limit = any(
                "enqueued token limit" in (e.message or "").lower()
                for e in errors
            )
            if token_limit and retries < MAX_RETRIES:
                retries += 1
                print(f"Token limit hit - waiting {RETRY_WAIT}s then resubmitting (attempt {retries}/{MAX_RETRIES})")
                time.sleep(RETRY_WAIT)
                batch_id = _resubmit(batch_id)
                print(f"Resubmitted as {batch_id}")
            elif retries >= MAX_RETRIES:
                print(f"Max retries ({MAX_RETRIES}) reached - skipping.\n")
                _remove_from_metadata(batch_id)
                break
            else:
                messages = [e.message for e in errors]
                print(f"Failed: {messages} - skipping.\n")
                _remove_from_metadata(batch_id)
                break

        elif status in ("cancelling", "cancelled", "expired"):
            print(f"Status '{status}' - skipping.\n")
            _remove_from_metadata(batch_id)
            break

        else:
            completed = job.request_counts.completed
            total = job.request_counts.total
            clear_output(wait=True)
            print(f"{file_name} | {batch_id}")
            print(f"[{status}] {completed}/{total} - next check in {POLL_INTERVAL}s...")
            time.sleep(POLL_INTERVAL)

print("All batches processed.")

In [ ]:
import uuid

from google import genai
from google.genai import types

In [ ]:
with open(D.find("leafs.json"), "r", encoding="utf-8") as f:
    leafs: list[dict] = json.load(f)

print(f"Total leafs: {len(leafs):,}")

In [ ]:
# Responses <-- gemini API
leaf_embed_responses: list[dict] = []

GEMINI_MODEL: str = "gemini-embedding-001"
TASK_TYPE: str = "SEMANTIC_SIMILARITY"

load_dotenv()

# Connect to Gemini
gemini_key: str | None = os.getenv("GEMINI_API_KEY")
if not gemini_key:
    raise ValueError("GEMINI_API_KEY is not set in environment variables.")
gemini_client = genai.Client(api_key=gemini_key)

In [ ]:
RPM: int = 3_000
TPM: int = 1_000_000
BATCH_LIMIT: int = 100
MIN_BATCH_INTERVAL: float = (1 / RPM) * 60

In [ ]:
SCHEME: str = "o200k_base"
enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

total_tokens: int = sum(len(enc.encode(leaf["text"])) for leaf in leafs)
print(f"Total tokens across all leafs: {total_tokens:,}")

In [ ]:
# Build batches of up to BATCH_LIMIT items each
gemini_batches: list[list[dict]] = []

for idx in range(0, len(leafs), BATCH_LIMIT):
    gemini_batches.append(leafs[idx:idx + BATCH_LIMIT])

print(f"Total batches: {len(gemini_batches):,}")

In [ ]:
SAFETY_BUFFER: int = 20

window_start: float = time.monotonic()
window_tokens: int = 0
window_requests: int = 0

for idx, batch in enumerate(gemini_batches):
    batch_start: float = time.monotonic()
    batch_tokens: int = sum(len(enc.encode(leaf["text"])) for leaf in batch)

    # Check if this batch would exceed the rolling window limits
    would_exceed_rpm: bool = (window_requests + 1) > RPM
    would_exceed_tpm: bool = (window_tokens + batch_tokens) > TPM
    window_elapsed: float = batch_start - window_start

    if window_elapsed < 60 and (would_exceed_rpm or would_exceed_tpm):
        wait_time: float = 60 - window_elapsed + SAFETY_BUFFER
        clear_output(wait=True)
        print(
            f"=== RATE LIMIT APPROACHING – sleeping {wait_time:.0f}s before batch "
            f"{idx + 1}/{len(gemini_batches)} ===\n"
            f"    RPM: {window_requests}/{RPM} | TPM: {window_tokens:,}/{TPM:,}"
        )
        time.sleep(wait_time)

        # Reset the sliding window
        window_start = time.monotonic()
        window_tokens = 0
        window_requests = 0

    # If the window has naturally expired (>60s), reset counters
    if (batch_start - window_start) >= 60:
        window_start = time.monotonic()
        window_tokens = 0
        window_requests = 0

    # Accumulate usage
    window_tokens += batch_tokens
    window_requests += 1

    clear_output(wait=True)
    print(
        f"Processing batch {idx + 1}/{len(gemini_batches)} "
        f"({len(batch)} items, {batch_tokens:,} tokens) | "
        f"Window RPM: {window_requests}/{RPM} | Window TPM: {window_tokens:,}/{TPM:,}"
    )

    # Call Gemini embed_content (batch of up to 100 texts)
    result = gemini_client.models.embed_content(
        model=GEMINI_MODEL,
        contents=[leaf["text"] for leaf in batch],
        config=types.EmbedContentConfig(
            task_type=TASK_TYPE,
            output_dimensionality=3072
        )
    )

    # Validate response count
    if len(result.embeddings) != len(batch):
        raise ValueError(
            f"Batch {idx}: expected {len(batch)} embeddings, "
            f"got {len(result.embeddings)}"
        )

    for jdx, embedding in enumerate(result.embeddings):
        leaf_embed_responses.append({
            **batch[jdx],
            "embedding": embedding.values,
            "qdrant-id": str(uuid.uuid5(uuid.NAMESPACE_DNS, batch[jdx]["uuid"]))
        })

    # Pace requests to stay under RPM even if TPM has room
    elapsed: float = time.monotonic() - batch_start
    sleep_time: float = MIN_BATCH_INTERVAL - elapsed
    if sleep_time > 0:
        time.sleep(sleep_time)

print(f"\nDone! Collected {len(leaf_embed_responses):,} embeddings.")

In [ ]:
output_path = D.processed / "leaf_embed.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(leaf_embed_responses, f, ensure_ascii=False, default=str)

print(f"Saved {len(leaf_embed_responses):,} leaf embeddings → {output_path}")

In [ ]:
import ijson

# Verifying if the output file is valid
with open(output_path, "r", encoding="utf-8") as f:
    first_object = next(ijson.items(f, "item"))

In [ ]:
first_object